# MR Protien Clinical Validation 

In [ ]:
protein_data = pd.read_csv("./data/roger/brain_heart_PA/original/olink2_data_renamed.tsv", sep="\t")

In [ ]:
for i in protein_data.columns:
    print(i)

In [ ]:
unet_t1 = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/percentiles_T1.csv", sep = ",")

In [ ]:
unet_t1 = unet_t1[unet_t1['Mean_T1'] >= 50]

In [ ]:
from datetime import datetime

def calculate_age(year_of_birth, month_of_birth, date_death = None, study_end_date='2022-10-18'):
    # Convert month name to number (1-12)
    month_dict = {
        'January': 1, 'February': 2, 'March': 3, 'April': 4,
        'May': 5, 'June': 6, 'July': 7, 'August': 8,
        'September': 9, 'October': 10, 'November': 11, 'December': 12
    }

    if pd.isna(date_death):
        reference_date = datetime.strptime(study_end_date, '%Y-%m-%d')
        reference_year = reference_date.year
        reference_month = reference_date.month
    else:
        # Patient died, use date of death
        try:
            reference_date = datetime.strptime(date_death, '%Y-%m-%d')
            reference_year = reference_date.year
            reference_month = reference_date.month
        except:
            # Handle invalid date format
            return None
    
    # Calculate basic age
    age = reference_year - year_of_birth
    
    # Adjust if birthday hasn't occurred yet this year
    if reference_month < month_dict[month_of_birth]:
        age -= 1
        
    return age

def estimate_mri_date(year_of_birth, month_of_birth, age_at_mri):
    """Estimate MRI date from birth info and age at MRI"""
    month_dict = {
        'January': 1, 'February': 2, 'March': 3, 'April': 4,
        'May': 5, 'June': 6, 'July': 7, 'August': 8,
        'September': 9, 'October': 10, 'November': 11, 'December': 12
    }
    
    mri_year = int(year_of_birth + age_at_mri)
    mri_month = month_dict[month_of_birth]
    
    # Use mid-month (15th) as approximation
    return pd.Timestamp(year=mri_year, month=mri_month, day=15)

In [ ]:
big_pheno_data['current_age'] = big_pheno_data.apply(
    lambda row: calculate_age(row['year_of_birth'], row['month_of_birth'], row['date_death']),
    axis=1
)
unet_t1["Patient_ID"] = [int(patient[:7]) for patient in unet_t1["Patient_ID"]]
merged_data = pd.merge(unet_t1, big_pheno_data, left_on = "Patient_ID", right_on = "id")

# Apply to all rows
merged_data['estimated_mri_date'] = merged_data.apply(
    lambda row: estimate_mri_date(row['year_of_birth'], 
                                   row['month_of_birth'], 
                                   row['age_at_mri']), 
    axis=1
)
# Calculate time using DATES not ages
study_end = pd.to_datetime('2022-10-18')
merged_data['date_death_dt'] = pd.to_datetime(merged_data['date_death'])  # or date_death?

In [ ]:
pheno_ids = set(big_pheno_data["IID"])
protein_ids = set(protein_data["eid"])

overlap = pheno_ids & protein_ids

print(f"Pheno IDs:   {len(pheno_ids)}")
print(f"Protein IDs: {len(protein_ids)}")
print(f"Overlap:     {len(overlap)}")
print(f"In pheno only:   {len(pheno_ids - protein_ids)}")
print(f"In protein only: {len(protein_ids - pheno_ids)}")

In [ ]:
merged_data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')
 
# ── 1. MERGE PROTEIN + PHENOTYPE DATA ────────────────────────────────────────
 
def prepare_analysis_df(merged_data, protein_data):
    """Merge protein data with phenotype data on FID/id."""
    proteins = ['ctss', 'ecm1', 'folh1', 'scgn']
    
    protein_subset = protein_data[['id'] + proteins].copy()
    
    analysis_df = merged_data.merge(
        protein_subset,
        left_on='FID_x',
        right_on='id',
        how='inner'
    )
    
    print(f"Analysis sample N: {len(analysis_df)}")
    print(f"Protein missingness:")
    for p in proteins:
        n_miss = analysis_df[p].isna().sum()
        print(f"  {p.upper()}: {n_miss} missing ({n_miss/len(analysis_df)*100:.1f}%)")
    
    return analysis_df
 
 
# ── 2. DISEASE ASSOCIATION ANALYSIS ──────────────────────────────────────────
 
def run_disease_associations(analysis_df, disease_groups, proteins=['ctss', 'ecm1', 'folh1', 'scgn']):
    """
    For each protein x disease group, compare protein levels
    in cases vs controls using linear regression.
    
    Returns DataFrame with beta, SE, p-value, n_cases for each combination.
    Note: proteins already regressed for age/sex so no additional covariates needed.
    """
    results = []
    
    all_disease_ids = set()
    for ids in disease_groups.values():
        all_disease_ids.update(ids)
    
    for disease_name, case_ids in disease_groups.items():
        # Cases: in disease group AND in analysis_df
        case_mask = analysis_df['FID_x'].isin(case_ids)
        n_cases = case_mask.sum()
        
        if n_cases < 10:
            print(f"WARNING: {disease_name} has only {n_cases} cases — skipping")
            continue
        
        # Controls: not in ANY disease group
        control_mask = ~analysis_df['FID_x'].isin(all_disease_ids)
        n_controls = control_mask.sum()
        
        analysis_subset = analysis_df[case_mask | control_mask].copy()
        analysis_subset['is_case'] = analysis_subset['FID_x'].isin(case_ids).astype(int)
        
        for protein in proteins:
            subset = analysis_subset[['is_case', protein]].dropna()
            
            if len(subset) < 20:
                continue
            
            # Linear regression: protein ~ case_status
            # (proteins already age/sex adjusted)
            try:
                model = smf.ols(f'{protein} ~ is_case', data=subset).fit()
                beta = model.params['is_case']
                se = model.bse['is_case']
                pval = model.pvalues['is_case']
                
                # Also compute simple mean difference for interpretability
                case_mean = subset[subset['is_case']==1][protein].mean()
                ctrl_mean = subset[subset['is_case']==0][protein].mean()
                
                results.append({
                    'disease': disease_name,
                    'protein': protein.upper(),
                    'beta': beta,
                    'se': se,
                    'ci_lower': beta - 1.96 * se,
                    'ci_upper': beta + 1.96 * se,
                    'pval': pval,
                    'n_cases': n_cases,
                    'n_controls': n_controls,
                    'case_mean': case_mean,
                    'ctrl_mean': ctrl_mean,
                })
            except Exception as e:
                print(f"Error for {protein} x {disease_name}: {e}")
    
    results_df = pd.DataFrame(results)
    
    # Bonferroni correction across all tests
    n_tests = len(results_df)
    results_df['pval_bonf'] = np.minimum(results_df['pval'] * n_tests, 1.0)
    results_df['significant'] = results_df['pval_bonf'] < 0.05
    results_df['nominal_sig'] = results_df['pval'] < 0.05
    
    return results_df
 
 
# ── 3. FOREST PLOT ────────────────────────────────────────────────────────────
 
def plot_forest(results_df, save_path='protein_disease_forest.pdf'):
    """
    Publication-ready forest plot.
    One panel per protein, diseases on y-axis, beta (NPX difference) on x-axis.
    """
    proteins = ['CTSS', 'ECM1', 'FOLH1', 'SCGN']
    
    # Color scheme: two biological axes
    # Matrix remodelling (CTSS, ECM1) vs fibroblast activation (FOLH1, SCGN)
    protein_colors = {
        'CTSS':  '#C0392B',   # deep red
        'ECM1':  '#E67E22',   # orange
        'FOLH1': '#2980B9',   # blue
        'SCGN':  '#27AE60',   # green
    }
    
    fig, axes = plt.subplots(1, 4, figsize=(18, 7), sharey=True)
    fig.patch.set_facecolor('white')
    
    # Get ordered disease list (by mean effect size across proteins for ordering)
    disease_order = (
        results_df.groupby('disease')['beta']
        .mean()
        .sort_values(ascending=True)
        .index.tolist()
    )
    
    for ax, protein in zip(axes, proteins):
        pdata = results_df[results_df['protein'] == protein].copy()
        pdata = pdata.set_index('disease').reindex(disease_order).reset_index()
        pdata = pdata.dropna(subset=['beta'])
        
        color = protein_colors[protein]
        y_positions = np.arange(len(pdata))
        
        ax.axvline(x=0, color='#555555', linewidth=0.8, linestyle='--', alpha=0.7)
        
        for i, row in pdata.iterrows():
            y = y_positions[list(pdata.index).index(i)] if i in pdata.index else i
            
            # Determine marker style by significance
            if row['significant']:
                marker_size = 8
                alpha = 1.0
                edge_color = color
                face_color = color
            elif row['nominal_sig']:
                marker_size = 7
                alpha = 0.8
                edge_color = color
                face_color = color
            else:
                marker_size = 6
                alpha = 0.4
                edge_color = color
                face_color = 'white'
            
            y_idx = list(pdata['disease']).index(row['disease'])
            
            # Error bar
            ax.plot(
                [row['ci_lower'], row['ci_upper']],
                [y_idx, y_idx],
                color=color, alpha=alpha, linewidth=1.5, solid_capstyle='round'
            )
            # Point estimate
            ax.scatter(
                row['beta'], y_idx,
                color=face_color, edgecolors=edge_color,
                s=marker_size**2, zorder=5, linewidths=1.5, alpha=alpha
            )
            
            # Add n_cases annotation
            ax.text(
                ax.get_xlim()[1] if ax.get_xlim()[1] != 0 else 0.5,
                y_idx, f"n={int(row['n_cases'])}",
                va='center', ha='left', fontsize=7, color='#888888'
            )
        
        ax.set_yticks(range(len(pdata)))
        ax.set_yticklabels(pdata['disease'], fontsize=10)
        ax.set_xlabel('β (NPX difference vs controls)', fontsize=10)
        ax.set_title(protein, fontsize=13, fontweight='bold', color=color, pad=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.tick_params(left=False)
        ax.grid(axis='x', alpha=0.2, linewidth=0.5)
        
        # Shade alternating rows for readability
        for j in range(0, len(pdata), 2):
            ax.axhspan(j - 0.5, j + 0.5, alpha=0.04, color='grey')
    
    # Legend
    legend_elements = [
        mpatches.Patch(color='#333333', label='Bonferroni significant'),
        mpatches.Patch(color='#333333', alpha=0.5, label='Nominally significant (p<0.05)'),
        mpatches.Patch(facecolor='white', edgecolor='#333333', label='Not significant'),
    ]
    fig.legend(
        handles=legend_elements,
        loc='lower center', ncol=3,
        frameon=False, fontsize=9,
        bbox_to_anchor=(0.5, -0.02)
    )
    
    # Axis labels for two biological groups
    fig.text(0.27, 1.01, 'Matrix remodelling', ha='center', fontsize=10,
             style='italic', color='#888888')
    fig.text(0.73, 1.01, 'Fibroblast activation signalling', ha='center', fontsize=10,
             style='italic', color='#888888')
    
    plt.suptitle(
        'Circulating protein levels by cardiac disease group\n(cases vs disease-free controls)',
        y=1.06, fontsize=12, fontweight='bold'
    )
    
    plt.tight_layout()
    #plt.savefig(save_path, bbox_inches='tight', dpi=300)
    #plt.savefig(save_path.replace('.pdf', '.png'), bbox_inches='tight', dpi=300)
    print(f"Saved: {save_path}")
    plt.show()
    
    return fig
 
 
# ── 4. SUMMARY TABLE ─────────────────────────────────────────────────────────
 
def make_summary_table(results_df, save_path='protein_disease_table.csv'):
    """Clean summary table for supplement."""
    table = results_df[[
        'protein', 'disease', 'n_cases', 'n_controls',
        'beta', 'se', 'ci_lower', 'ci_upper', 'pval', 'pval_bonf', 'significant'
    ]].copy()
    
    table['beta'] = table['beta'].round(3)
    table['se'] = table['se'].round(3)
    table['ci_lower'] = table['ci_lower'].round(3)
    table['ci_upper'] = table['ci_upper'].round(3)
    table['pval'] = table['pval'].apply(lambda x: f"{x:.2e}")
    table['pval_bonf'] = table['pval_bonf'].apply(lambda x: f"{x:.2e}")
    
    table = table.sort_values(['protein', 'pval'])
    #table.to_csv(save_path, index=False)
    print(f"Saved: {save_path}")
    
    return table
 

# print("=== Preparing analysis dataset ===")
# analysis_df = prepare_analysis_df(merged_data, protein_data)

# print("\n=== Running disease associations ===")
# print("Disease group sizes:")
# for name, ids in disease_groups.items():
#     n_in_sample = analysis_df['FID_x'].isin(ids).sum()
#     print(f"  {name}: {len(ids)} total, {n_in_sample} in protein subset")

# results_df = run_disease_associations(analysis_df, disease_groups)

# print(f"\n=== Results summary ===")
# print(f"Total tests: {len(results_df)}")
# print(f"Bonferroni significant: {results_df['significant'].sum()}")
# print(f"Nominally significant: {results_df['nominal_sig'].sum()}")
# print("\nSignificant associations:")
# sig = results_df[results_df['significant']].sort_values('pval')
# print(sig[['protein', 'disease', 'beta', 'pval', 'pval_bonf', 'n_cases']].to_string())

# print("\n=== Generating forest plot ===")
# fig = plot_forest(results_df)

# print("\n=== Saving summary table ===")
# #table = make_summary_table(results_df, save_path='/mnt/user-data/outputs/protein_disease_table.csv')


In [ ]:
import pandas as pd

# ── LOAD PWAS RESULTS ─────────────────────────────────────────────────────────
t1_pwas = pd.read_csv(
    './data/shriya/pwas_results/'
    'cardiac_pwas_results_T1_percentiles_HHregressed (3).tsv',
    sep='\t'
)

vae_pwas = pd.read_csv(
    './data/shriya/pwas_results/'
    'cardiac_pwas_results_VAE_dimensions_notBMIregressed (3).tsv',
    sep='\t'
)

# ── COMBINE AND STANDARDISE ───────────────────────────────────────────────────
pwas_results = pd.concat([t1_pwas, vae_pwas], ignore_index=True)
pwas_results.columns = pwas_results.columns.str.strip()
pwas_results['Protein'] = pwas_results['Protein'].str.lower().str.strip()

print(f"Total associations: {len(pwas_results)}")
print(f"Significant associations: {pwas_results['Significant'].sum()}")
print(f"Unique proteins (all): {pwas_results['Protein'].nunique()}")
print(f"Unique proteins (significant): {pwas_results[pwas_results['Significant']==True]['Protein'].nunique()}")
print(f"Unique phenotypes: {pwas_results['PA_Phenotype'].nunique()}")
print(f"\nColumns: {pwas_results.columns.tolist()}")
print(pwas_results.head())

# ── GET SIGNIFICANT PROTEINS ──────────────────────────────────────────────────
sig_proteins = (
    pwas_results['Protein']
    #pwas_results[pwas_results['Significant'] == True]['Protein']
    .unique()
    .tolist()
)

# Keep only those present in protein_data
sig_proteins = [p for p in sig_proteins if p in protein_data.columns]
print(f"\nSignificant proteins on panel: {len(sig_proteins)}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import shap

In [ ]:
# ── 2. MERGE PROTEIN + T1 DATA ────────────────────────────────────────────────
df = protein_data[['eid'] + sig_proteins].merge(
    unet_t1[['Patient_ID', 'Mean_T1', 'T1_25th_Percentile', 'T1_75th_Percentile', "T1_99th_Percentile"]],
    left_on='eid', right_on='Patient_ID', how='inner'
).dropna()

print(f"Analysis N: {len(df)}")

# ── 3. DEFINE FEATURES AND TARGET ─────────────────────────────────────────────
# Change target to whichever T1 phenotype you want to predict
TARGET = 'Mean_T1'

X = df[sig_proteins].values
y = df[TARGET].values
feature_names = sig_proteins

# ── 4. TRAIN/TEST SPLIT ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# ── 5. FIT MODELS ─────────────────────────────────────────────────────────────
# Random Forest
rf = RandomForestRegressor(n_estimators=500, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
print(f"\nRandom Forest — R²: {rf_r2:.3f}, RMSE: {rf_rmse:.3f}")

# XGBoost
xgb = XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, random_state=42,
                   n_jobs=-1, verbosity=0)
xgb.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False)
xgb_pred = xgb.predict(X_test)
xgb_r2 = r2_score(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
print(f"XGBoost     — R²: {xgb_r2:.3f}, RMSE: {xgb_rmse:.3f}")

# ── 6. FEATURE IMPORTANCE — BUILT-IN ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, model, name, color in zip(
    axes,
    [rf, xgb],
    ['Random Forest', 'XGBoost'],
    ['#2980B9', '#C0392B']
):
    importances = model.feature_importances_
    idx = np.argsort(importances)[-30:]  # top 30
    
    ax.barh(
        [feature_names[i].upper() for i in idx],
        importances[idx],
        color=color, alpha=0.8
    )
    ax.set_xlabel('Feature Importance', fontsize=11)
    ax.set_title(f'{name}\nR²={r2_score(y_test, model.predict(X_test)):.3f}',
                 fontsize=13, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle(f'Top 30 Proteins Predicting {TARGET}', fontsize=14, fontweight='bold')
plt.tight_layout()
#plt.savefig('feature_importance_builtin.png', dpi=300, bbox_inches='tight')
plt.show()

# ── 7. SHAP VALUES (XGBoost) ──────────────────────────────────────────────────
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

In [ ]:
# Summary plot — top 30 proteins
plt.figure()
shap.summary_plot(
    shap_values, X_test,
    feature_names=[f.upper() for f in feature_names],
    max_display=30,
    show=False
)
plt.title(f'SHAP Values — XGBoost predicting {TARGET}', fontsize=12)
plt.tight_layout()
#plt.savefig('shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

# ── 8. SAVE FEATURE IMPORTANCE TABLE ──────────────────────────────────────────
importance_df = pd.DataFrame({
    'protein': feature_names,
    'rf_importance': rf.feature_importances_,
    'xgb_importance': xgb.feature_importances_,
    'shap_mean_abs': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_mean_abs', ascending=False)

#importance_df.to_csv('protein_feature_importance.csv', index=False)
print("\nTop 10 proteins by SHAP:")
print(importance_df.head(10)[['protein', 'rf_importance', 'xgb_importance', 'shap_mean_abs']].to_string())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, model, name, color, pred in zip(
    axes,
    [rf, xgb],
    ['Random Forest', 'XGBoost'],
    ['#2980B9', '#C0392B'],
    [rf_pred, xgb_pred]
):
    r2 = r2_score(y_test, pred)
    
    # Scatter
    ax.scatter(y_test, pred, alpha=0.3, s=10, color=color, rasterized=True)
    
    # Perfect prediction line
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, 'k--', linewidth=1, alpha=0.7, label='Perfect prediction')
    
    # Regression line through predictions
    m, b = np.polyfit(y_test, pred, 1)
    ax.plot(lims, [m*x + b for x in lims], color='red', 
            linewidth=1.5, alpha=0.8, label=f'Fit (slope={m:.2f})')
    
    ax.set_xlabel('Actual T1 (regressed)', fontsize=11)
    ax.set_ylabel('Predicted T1', fontsize=11)
    ax.set_title(f'{name}\nR²={r2:.3f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Predicted vs Actual Mean T1', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('prediction_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
CUTOFF = 1000
T1_COL = 'Mean_T1'

df = protein_data[['eid'] + sig_proteins].merge(
    unet_t1,
    left_on='eid', right_on='Patient_ID',
    how='inner'
).dropna()

df = df.merge(
    latent_dimensions,
    left_on='eid', right_on='IID',
    how='inner'
).dropna()

latent_dimensions

df['elevated_T1'] = (df[T1_COL] > CUTOFF).astype(int)

n_cases = df['elevated_T1'].sum()
n_controls = (df['elevated_T1'] == 0).sum()
print(f"Total N: {len(df)}")
print(f"Elevated T1 (cases): {n_cases} ({n_cases/len(df)*100:.1f}%)")
print(f"Normal T1 (controls): {n_controls} ({n_controls/len(df)*100:.1f}%)")

In [ ]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score, RandomizedSearchCV)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import numpy as np

# ── 2. FEATURES AND TARGET ────────────────────────────────────────────────────
X = df[sig_proteins].values
y = df['elevated_T1'].values
feature_names = sig_proteins

# ── 3. TRAIN/TEST SPLIT (stratified) ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # ensures same case/control ratio in train and test
)
print(f"\nTrain: {X_train.shape[0]} (cases: {y_train.sum()})")
print(f"Test:  {X_test.shape[0]} (cases: {y_test.sum()})")

# ── 4. CLASS WEIGHT FOR IMBALANCE ────────────────────────────────────────────
scale_pos_weight = n_controls / n_cases  # handles class imbalance
print(f"\nScale pos weight: {scale_pos_weight:.2f}")

# ── 5. FIT MODELS ─────────────────────────────────────────────────────────────
# XGBoost
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    eval_metric='auc'
)
xgb.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# ── 6. EVALUATE ───────────────────────────────────────────────────────────────
xgb_proba = xgb.predict_proba(X_test)[:, 1]
rf_proba = rf.predict_proba(X_test)[:, 1]

xgb_auc = roc_auc_score(y_test, xgb_proba)
rf_auc = roc_auc_score(y_test, rf_proba)

print(f"\nXGBoost AUC: {xgb_auc:.3f}")
print(f"Random Forest AUC: {rf_auc:.3f}")

print("\nXGBoost Classification Report:")
print(classification_report(y_test, xgb.predict(X_test)))

# ── 7. ROC CURVES ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

for proba, auc, name, color in zip(
    [xgb_proba, rf_proba],
    [xgb_auc, rf_auc],
    ['XGBoost', 'Random Forest'],
    ['#C0392B', '#2980B9']
):
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(f'ROC Curve — Elevated T1 (>{CUTOFF}ms)\nProteins as features',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('roc_curve_binary_T1.png', dpi=300, bbox_inches='tight')
plt.show()

# ── 8. FEATURE IMPORTANCE ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, model, name, color in zip(
    axes,
    [rf, xgb],
    ['Random Forest', 'XGBoost'],
    ['#2980B9', '#C0392B']
):
    importances = model.feature_importances_
    idx = np.argsort(importances)[-30:]
    ax.barh(
        [feature_names[i].upper() for i in idx],
        importances[idx],
        color=color, alpha=0.8
    )
    ax.set_xlabel('Feature Importance', fontsize=11)
    ax.set_title(f'{name} (AUC={roc_auc_score(y_test, model.predict_proba(X_test)[:,1]):.3f})',
                 fontsize=12, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle(f'Top 30 Proteins — Elevated T1 >{CUTOFF}ms',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance_binary.png', dpi=300, bbox_inches='tight')
plt.show()

# ── 9. SHAP VALUES ────────────────────────────────────────────────────────────
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

plt.figure()
shap.summary_plot(
    shap_values, X_test,
    feature_names=[f.upper() for f in feature_names],
    max_display=30,
    show=False
)
plt.title(f'SHAP Values — XGBoost, Elevated T1 >{CUTOFF}ms', fontsize=11)
plt.tight_layout()
plt.savefig('shap_binary_T1.png', dpi=300, bbox_inches='tight')
plt.show()

# ── 10. SAVE IMPORTANCE TABLE ─────────────────────────────────────────────────
importance_df = pd.DataFrame({
    'protein': feature_names,
    'rf_importance': rf.feature_importances_,
    'xgb_importance': xgb.feature_importances_,
    'shap_mean_abs': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_mean_abs', ascending=False)

importance_df.to_csv('binary_T1_protein_importance.csv', index=False)
print("\nTop 10 proteins by SHAP:")
print(importance_df.head(10).to_string())

In [ ]:
def train_binary_t1_model(df, sig_proteins, phenotype, cutoff_percentile=0.75):
    """
    Train XGBoost and Random Forest binary classifiers to predict
    high vs low T1 phenotype using PWAS-significant proteins.
    
    Parameters:
    -----------
    df : DataFrame with protein columns + T1 phenotype columns
    sig_proteins : list of protein column names
    phenotype : str, T1 column to use (e.g. 'Mean_T1', 'T1_75th_Percentile')
    cutoff_percentile : float, percentile threshold (default 0.75)
    
    Returns:
    --------
    results : dict with models, AUCs, importance_df, shap_values
    """
    from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score, roc_curve, classification_report
    from xgboost import XGBClassifier
    import shap
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    # ── 1. CREATE BINARY LABEL ────────────────────────────────────────────────
    threshold = df[phenotype].quantile(cutoff_percentile)
    print(f"Phenotype:  {phenotype}")
    print(f"Cutoff:     {cutoff_percentile*100:.0f}th percentile = {threshold:.1f}ms")

    df_binary = df[[phenotype] + sig_proteins].copy()
    df_binary['label'] = (df_binary[phenotype] > threshold).astype(int)
    df_binary = df_binary.dropna()

    n_cases    = df_binary['label'].sum()
    n_controls = (df_binary['label'] == 0).sum()
    print(f"Total N:    {len(df_binary)}")
    print(f"Cases:      {n_cases} ({n_cases/len(df_binary)*100:.1f}%)")
    print(f"Controls:   {n_controls} ({n_controls/len(df_binary)*100:.1f}%)")

    if n_cases < 20:
        print("WARNING: fewer than 20 cases — model will not be reliable")
        return None

    # ── 2. SPLIT ──────────────────────────────────────────────────────────────
    X = df_binary[sig_proteins].values
    y = df_binary['label'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"\nTrain: {X_train.shape[0]} | Test: {X_test.shape[0]}")

    # ── 3. NESTED CV HYPERPARAMETER OPTIMISATION ──────────────────────────────────
    # Inner CV: hyperparameter search on training data only
    # Outer CV: performance estimation (run after best params found)

    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    outer_cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # XGBoost hyperparameter grid
    xgb_param_grid = {
        'n_estimators':    [300, 500, 700],
        'max_depth':       [3, 4, 5, 6],
        'learning_rate':   [0.01, 0.05, 0.1],
        'subsample':       [0.7, 0.8, 0.9],
        'colsample_bytree':[0.7, 0.8, 0.9],
        'min_child_weight':[1, 3, 5]
    }

    # Random Forest hyperparameter grid
    rf_param_grid = {
        'n_estimators': [300, 500, 700],
        'max_depth':    [4, 6, 8, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
        'max_features': ['sqrt', 'log2']
    }

    print("\nOptimising XGBoost hyperparameters...")
    xgb_search = RandomizedSearchCV(
        XGBClassifier(
            scale_pos_weight=n_controls/n_cases,
            random_state=42, n_jobs=-1, verbosity=0, eval_metric='auc'
        ),
        param_distributions=xgb_param_grid,
        n_iter=20,                    # number of random combinations to try
        scoring='roc_auc',
        cv=inner_cv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    xgb_search.fit(X_train, y_train)
    print(f"Best XGBoost params: {xgb_search.best_params_}")
    print(f"Best inner CV AUC:   {xgb_search.best_score_:.3f}")

    print("\nOptimising Random Forest hyperparameters...")
    rf_search = RandomizedSearchCV(
        RandomForestClassifier(
            class_weight='balanced', random_state=42, n_jobs=-1
        ),
        param_distributions=rf_param_grid,
        n_iter=20,
        scoring='roc_auc',
        cv=inner_cv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    rf_search.fit(X_train, y_train)
    print(f"Best RF params:    {rf_search.best_params_}")
    print(f"Best inner CV AUC: {rf_search.best_score_:.3f}")

    # ── 4. FIT FINAL MODELS WITH BEST PARAMS ON FULL TRAINING SET ────────────────
    xgb = xgb_search.best_estimator_
    rf  = rf_search.best_estimator_

    # Refit on full training set with best params
    xgb.fit(X_train, y_train)
    rf.fit(X_train, y_train)

    # ── 4. EVALUATE ───────────────────────────────────────────────────────────
    xgb_proba = xgb.predict_proba(X_test)[:, 1]
    rf_proba  = rf.predict_proba(X_test)[:, 1]
    xgb_auc   = roc_auc_score(y_test, xgb_proba)
    rf_auc    = roc_auc_score(y_test, rf_proba)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_aucs = cross_val_score(xgb, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

    def compute_metrics(y_true, y_pred, y_proba, model_name):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        sensitivity  = tp / (tp + fn) if (tp + fn) > 0 else 0  # recall
        specificity  = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv          = tp / (tp + fp) if (tp + fp) > 0 else 0  # precision
        npv          = tn / (tn + fn) if (tn + fn) > 0 else 0
        accuracy     = (tp + tn) / (tp + tn + fp + fn)
        f1           = 2 * (ppv * sensitivity) / (ppv + sensitivity) if (ppv + sensitivity) > 0 else 0
        auc          = roc_auc_score(y_true, y_proba)

        # Bootstrap CI for AUC
        bootstrap_aucs = []
        for _ in range(1000):
            idx = np.random.choice(len(y_true), len(y_true), replace=True)
            if len(np.unique(y_true[idx])) == 2:
                bootstrap_aucs.append(roc_auc_score(y_true[idx], y_proba[idx]))
        ci_lower = np.percentile(bootstrap_aucs, 2.5)
        ci_upper = np.percentile(bootstrap_aucs, 97.5)

        print(f"\n{'='*50}")
        print(f"{model_name} — Performance Metrics")
        print(f"{'='*50}")
        print(f"AUC:              {auc:.3f} (95% CI {ci_lower:.3f}–{ci_upper:.3f})")
        print(f"Accuracy:         {accuracy:.3f}")
        print(f"Sensitivity:      {sensitivity:.3f}  (TP={tp}, FN={fn})")
        print(f"Specificity:      {specificity:.3f}  (TN={tn}, FP={fp})")
        print(f"PPV:              {ppv:.3f}")
        print(f"NPV:              {npv:.3f}")
        print(f"F1 Score:         {f1:.3f}")

        return {
            'auc': auc, 'ci_lower': ci_lower, 'ci_upper': ci_upper,
            'sensitivity': sensitivity, 'specificity': specificity,
            'ppv': ppv, 'npv': npv, 'accuracy': accuracy, 'f1': f1,
            'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn
        }
    
    fpr, tpr, thresholds = roc_curve(y_test, xgb_proba)
    youden_idx  = np.argmax(tpr - fpr)
    optimal_thr = thresholds[youden_idx]
    print(f"\nOptimal threshold (Youden's J): {optimal_thr:.3f}")

    # Recompute with optimal threshold
    xgb_pred_optimal = (xgb_proba >= optimal_thr).astype(int)
    xgb_metrics_opt  = compute_metrics(y_test, xgb_pred_optimal, xgb_proba, 'XGBoost (optimal threshold)')

    # Run for both models
    xgb_metrics = compute_metrics(y_test, xgb.predict(X_test), xgb_proba, 'XGBoost (0.5 threshold)')
    rf_metrics  = compute_metrics(y_test, rf.predict(X_test),  rf_proba,  'Random Forest (0.5 threshold)')

    print(f"\n5-fold CV AUC: {cv_aucs.mean():.3f} ± {cv_aucs.std():.3f}")
    print("\nXGBoost Classification Report:")
    print(classification_report(y_test, xgb.predict(X_test)))

    # ── 5. ROC CURVE ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 6))
    for proba, auc, name, color in zip(
        [xgb_proba, rf_proba],
        [xgb_auc, rf_auc],
        ['XGBoost', 'Random Forest'],
        ['#C0392B', '#2980B9']
    ):
        fpr, tpr, _ = roc_curve(y_test, proba)
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{name} (AUC={auc:.3f})')

    ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(
        f'ROC Curve — {phenotype}\n'
        f'>{cutoff_percentile*100:.0f}th percentile ({threshold:.0f}ms)',
        fontsize=11, fontweight='bold'
    )
    ax.legend(fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(f'roc_{phenotype}_{int(cutoff_percentile*100)}pct.png',
                dpi=300, bbox_inches='tight')
    plt.show()

    # ── 6. FEATURE IMPORTANCE ─────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    for ax, model, name, color in zip(
        axes, [rf, xgb], ['Random Forest', 'XGBoost'], ['#2980B9', '#C0392B']
    ):
        importances = model.feature_importances_
        idx = np.argsort(importances)[-30:]
        ax.barh(
            [sig_proteins[i].upper() for i in idx],
            importances[idx], color=color, alpha=0.8
        )
        ax.set_xlabel('Feature Importance', fontsize=11)
        ax.set_title(
            f'{name} (AUC={roc_auc_score(y_test, model.predict_proba(X_test)[:,1]):.3f})',
            fontsize=12, fontweight='bold'
        )
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle(
        f'Top 30 Proteins — {phenotype} >{cutoff_percentile*100:.0f}th percentile',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(f'importance_{phenotype}_{int(cutoff_percentile*100)}pct.png',
                dpi=300, bbox_inches='tight')
    plt.show()

    # ── 7. SHAP ───────────────────────────────────────────────────────────────
    explainer  = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_test)

    plt.figure()
    shap.summary_plot(
        shap_values, X_test,
        feature_names=[p.upper() for p in sig_proteins],
        max_display=30, show=False
    )
    plt.title(f'SHAP — {phenotype} >{cutoff_percentile*100:.0f}th percentile',
              fontsize=11)
    plt.tight_layout()
    plt.savefig(f'shap_{phenotype}_{int(cutoff_percentile*100)}pct.png',
                dpi=300, bbox_inches='tight')
    plt.show()

    # ── 8. IMPORTANCE TABLE ───────────────────────────────────────────────────
    importance_df = pd.DataFrame({
        'protein':        sig_proteins,
        'rf_importance':  rf.feature_importances_,
        'xgb_importance': xgb.feature_importances_,
        'shap_mean_abs':  np.abs(shap_values).mean(axis=0)
    }).sort_values('shap_mean_abs', ascending=False)

    print("\nTop 10 proteins by SHAP:")
    print(importance_df.head(10)[['protein', 'xgb_importance', 'shap_mean_abs']].to_string())

    return {
        'xgb': xgb, 'rf': rf,
        'xgb_auc': xgb_auc, 'rf_auc': rf_auc,
        'cv_auc_mean': cv_aucs.mean(), 'cv_auc_std': cv_aucs.std(),
        'importance_df': importance_df,
        'shap_values': shap_values,
        'threshold': threshold,
        'y_test': y_test,        # ADD
        'xgb_proba': xgb_proba  # ADD
    }

In [ ]:
df

In [ ]:
train_binary_t1_model(df, sig_proteins, 'Latent_11', cutoff_percentile=0.25)

In [ ]:
train_binary_t1_model(df, sig_proteins, 'Latent_5', cutoff_percentile=0.75)

In [ ]:
model_info_75p = train_binary_t1_model(df, sig_proteins, 'T1_75th_Percentile', cutoff_percentile=0.75)

In [ ]:
train_binary_t1_model(df, sig_proteins, 'Latent_1', cutoff_percentile=0.25)

In [ ]:
train_binary_t1_model(df, sig_proteins, 'T1_99th_Percentile', cutoff_percentile=0.5)

In [ ]:
df

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def biomarker_panel_lasso(phenotype, cutoff_percentile, top10_proteins):
    """
    Fit LASSO on top 10 XGBoost SHAP proteins.
    Reports coefficients, CV AUC, test AUC, and ROC curve.
    
    Parameters:
    -----------
    phenotype        : str   — T1 or latent column
    cutoff_percentile: float — percentile threshold
    top10_proteins   : list  — top 10 proteins from XGBoost SHAP
    """
    print(f"\n{'='*60}")
    print(f"Phenotype: {phenotype} @ {cutoff_percentile*100:.0f}th percentile")
    print(f"{'='*60}")

    # ── 1. BUILD DATASET ──────────────────────────────────────────
    
    unet_t1_dedup = unet_t1.groupby('Patient_ID').mean().reset_index()

    df = protein_data[['eid'] + top10_proteins].merge(
        unet_t1_dedup, left_on='eid', right_on='Patient_ID', how='inner'
    ).merge(
        latent_dimensions, left_on='eid', right_on='IID', how='inner'
    ).dropna()

    threshold   = df[phenotype].quantile(cutoff_percentile)
    df['label'] = (df[phenotype] > threshold).astype(int)

    n_cases    = df['label'].sum()
    n_controls = (df['label'] == 0).sum()
    print(f"N={len(df)} | Cases={n_cases} ({n_cases/len(df)*100:.1f}%) | "
          f"Controls={n_controls} ({n_controls/len(df)*100:.1f}%)")

    X = df[top10_proteins].values
    y = df['label'].values

    # ── 2. TRAIN/TEST SPLIT ───────────────────────────────────────
    X_dev, X_test, y_dev, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # ── 3. SCALE ON DEV ONLY ──────────────────────────────────────
    scaler        = StandardScaler()
    X_dev_scaled  = scaler.fit_transform(X_dev)
    X_test_scaled = scaler.transform(X_test)

    # ── 4. FIT LASSO ──────────────────────────────────────────────
    lasso = LogisticRegressionCV(
        Cs=20, cv=5, penalty='l1', solver='liblinear',
        scoring='roc_auc', random_state=42, n_jobs=-1,
        class_weight='balanced'
    )
    lasso.fit(X_dev_scaled, y_dev)

    # ── 5. CV AUC ON DEV ─────────────────────────────────────────
    cv_aucs = cross_val_score(
        lasso, X_dev_scaled, y_dev,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc', n_jobs=-1
    )

    # ── 6. TEST AUC ───────────────────────────────────────────────
    test_proba = lasso.predict_proba(X_test_scaled)[:, 1]
    test_auc   = roc_auc_score(y_test, test_proba)

    # Bootstrap CI
    bootstrap_aucs = []
    for _ in range(1000):
        idx = np.random.choice(len(y_test), len(y_test), replace=True)
        if len(np.unique(y_test[idx])) == 2:
            bootstrap_aucs.append(roc_auc_score(y_test[idx], test_proba[idx]))
    ci_lower = np.percentile(bootstrap_aucs, 2.5)
    ci_upper = np.percentile(bootstrap_aucs, 97.5)

    print(f"Dev CV AUC:  {cv_aucs.mean():.3f} ± {cv_aucs.std():.3f}")
    print(f"Test AUC:    {test_auc:.3f} (95% CI {ci_lower:.3f}–{ci_upper:.3f})")
    
    # Optimal threshold via Youden's J
    fpr_arr, tpr_arr, thresholds = roc_curve(y_test, test_proba)
    youden_idx  = np.argmax(tpr_arr - fpr_arr)
    optimal_thr = thresholds[youden_idx]
    test_pred_opt  = (test_proba >= optimal_thr).astype(int)
    tn_o, fp_o, fn_o, tp_o = confusion_matrix(y_test, test_pred_opt).ravel()

    tn, fp, fn, tp = confusion_matrix(y_test, test_pred_opt).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv         = tn / (tn + fn) if (tn + fn) > 0 else 0
    accuracy    = (tp + tn) / (tp + tn + fp + fn)
    f1          = (2 * ppv * sensitivity / (ppv + sensitivity)
                   if (ppv + sensitivity) > 0 else 0)
    print(f"\n{'='*50}")
    print(f"LASSO — Performance Metrics")
    print(f"{'='*50}")
    print(f"Dev CV AUC:       {cv_aucs.mean():.3f} ± {cv_aucs.std():.3f}")
    print(f"Test AUC:         {test_auc:.3f} (95% CI {ci_lower:.3f}–{ci_upper:.3f})")
    print(f"\n--- Default threshold (0.5) ---")
    print(f"Accuracy:         {accuracy:.3f}")
    print(f"Sensitivity:      {sensitivity:.3f}  (TP={tp}, FN={fn})")
    print(f"Specificity:      {specificity:.3f}  (TN={tn}, FP={fp})")
    print(f"PPV:              {ppv:.3f}")
    print(f"NPV:              {npv:.3f}")
    print(f"F1 Score:         {f1:.3f}")
    print(f"\n--- Optimal threshold (Youden's J = {optimal_thr:.3f}) ---")
    print(f"Sensitivity:      {tp_o/(tp_o+fn_o):.3f}  (TP={tp_o}, FN={fn_o})")
    print(f"Specificity:      {tn_o/(tn_o+fp_o):.3f}  (TN={tn_o}, FP={fp_o})")
    print(f"PPV:              {tp_o/(tp_o+fp_o) if (tp_o+fp_o)>0 else 0:.3f}")
    print(f"NPV:              {tn_o/(tn_o+fn_o) if (tn_o+fn_o)>0 else 0:.3f}")

    # ── 7. COEFFICIENTS ───────────────────────────────────────────
    coef_df = pd.DataFrame({
        'protein':     top10_proteins,
        'coefficient': lasso.coef_[0]
    }).assign(
        abs_coef   = lambda x: x['coefficient'].abs(),
        direction  = lambda x: x['coefficient'].apply(
            lambda c: '↑ risk' if c > 0 else '↓ risk' if c < 0 else 'excluded'
        )
    ).sort_values('abs_coef', ascending=False)

    print(f"\nLASSO coefficients (standardised):")
    print(coef_df[['protein', 'coefficient', 'direction']].to_string())

#     # ── 8. ROC CURVE ─────────────────────────────────────────────
#     fig, ax = plt.subplots(figsize=(6, 6))
#     fpr, tpr, _ = roc_curve(y_test, test_proba)
#     ax.plot(fpr, tpr, color='#C0392B', linewidth=2,
#             label=f'LASSO 10-protein panel\nAUC={test_auc:.3f} '
#                   f'(95% CI {ci_lower:.3f}–{ci_upper:.3f})')
#     ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.4, label='Random')
#     ax.set_xlabel('False Positive Rate', fontsize=12)
#     ax.set_ylabel('True Positive Rate', fontsize=12)
#     ax.set_title(f'{phenotype} >{cutoff_percentile*100:.0f}th percentile\n'
#                  f'10-protein LASSO panel',
#                  fontsize=11, fontweight='bold')
#     ax.legend(fontsize=9)
#     ax.spines['top'].set_visible(False)
#     ax.spines['right'].set_visible(False)
#     plt.tight_layout()
#     plt.savefig(f'roc_lasso10_{phenotype}_{int(cutoff_percentile*100)}pct.png',
#                 dpi=300, bbox_inches='tight')
#     plt.show()
    
    # ── 8. ROC CURVE WITH NT-proBNP OVERLAY ──────────────────────
        
    # Build BNP dataset independently — only needs eid, nppb, and phenotype
    df_bnp = protein_data[['eid', 'nppb']].merge(
        unet_t1_dedup, left_on='eid', right_on='Patient_ID', how='inner'
    ).merge(
        latent_dimensions, left_on='eid', right_on='IID', how='inner'
    ).dropna(subset=['nppb', phenotype])
    
    df_bnp['label'] = (df_bnp[phenotype] > threshold).astype(int)
    
    X_bnp = df_bnp['nppb'].values.reshape(-1, 1)
    y_bnp = df_bnp['label'].values
    
    _, X_bnp_test, _, y_bnp_test = train_test_split(
        X_bnp, y_bnp, test_size=0.2, random_state=42, stratify=y_bnp
    )
    bnp_scores = X_bnp_test[:, 0]

    auc_bnp = roc_auc_score(y_bnp_test, bnp_scores)
    fpr_bnp, tpr_bnp, _ = roc_curve(y_bnp_test, bnp_scores)
    
    bnp_bootstrap = []
    for _ in range(1000):
        idx = np.random.choice(len(y_bnp_test), len(y_bnp_test), replace=True)
        if len(np.unique(y_bnp_test[idx])) == 2:
            bnp_bootstrap.append(roc_auc_score(y_bnp_test[idx], bnp_scores[idx]))
    ci_bnp_lo = np.percentile(bnp_bootstrap, 2.5)
    ci_bnp_hi = np.percentile(bnp_bootstrap, 97.5)

# recompute explicitly to avoid variable collision
    fpr_lasso, tpr_lasso, _ = roc_curve(y_test, test_proba)
    fpr_bnp, tpr_bnp, _ = roc_curve(y_bnp_test, bnp_scores)

    fig, ax = plt.subplots(figsize=(5, 5))
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'DejaVu Serif'],
        'axes.spines.top': False,
        'axes.spines.right': False,
        'pdf.fonttype': 42,
    })

    ax.plot(fpr_lasso, tpr_lasso, color='#c0392b', lw=2,
            label=f'LASSO 10-protein panel\nAUC = {test_auc:.3f} '
                  f'(95% CI {ci_lower:.3f}–{ci_upper:.3f})')
    ax.plot(fpr_bnp, tpr_bnp, color='#2471a3', lw=2, linestyle='--',
            label=f'NT-proBNP alone\nAUC = {auc_bnp:.3f} '
                  f'(95% CI {ci_bnp_lo:.3f}–{ci_bnp_hi:.3f})')
    ax.plot([0, 1], [0, 1], color='#aaaaaa', lw=1, linestyle=':',
            label='Random')

    ax.set_xlabel('1 − Specificity', fontsize=11)
    ax.set_ylabel('Sensitivity', fontsize=11)
    ax.set_title('B', fontsize=13, fontweight='bold', loc='left')
    ax.set_title('T1 75th percentile — LASSO panel vs NT-proBNP',
                 fontsize=10, loc='center')
    ax.legend(fontsize=8.5, frameon=False, loc='lower right')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, color='#aaaaaa')
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.savefig(f'panel_B_roc_overlay_{phenotype}.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(f'panel_B_roc_overlay_{phenotype}.png', dpi=300, bbox_inches='tight')
    plt.show()

    return {
        'phenotype':  phenotype,
        'test_auc':   test_auc,
        'ci_lower':   ci_lower,
        'ci_upper':   ci_upper,
        'cv_auc':     cv_aucs.mean(),
        'cv_std':     cv_aucs.std(),
        'coef_df':    coef_df,
        'lasso':      lasso,
        'scaler':     scaler,
        'threshold':  threshold
    }


In [ ]:
# ── DEFINE TOP 10 PROTEINS PER PHENOTYPE ─────────────────────────────────────
top10_latent5 = ['cntn4', 'pdgfc', 'agr3', 'ctss', 'klrd1',
                 'robo1', 'plxnb2', 'gzmb', 'spink1', 'il13ra1']

top10_latent11 = ['klk6', 'pgf', 'ddx58', 'lrpap1', 'itih3', 'bst2',
                  'ncam2', 'traf2', 'ggt1', 'havcr1']

top10_latent1 = ['ccl3', 'pdgfc', 'serpina11', 'ces3', 'efemp1',
                 'il17rb', 'nadk', 'cntn4', 'wfikkn2', 's100a4']

# top10_t1_75p = ['fap', 'cd38', 'il1rl1', 'ren', 'angpt2',
#                 'igfbp1', 'clec14a', 'wnt9a', 'ptprm', 'mmp8']

top10_t1_75p = ['fap', 'igfbp1', 'il1rl1', 'ren', 'sele',
                'cd38', 'folr1', 'gzmh', 'dcbld2', 'st3gal1']

# ── RUN ALL FOUR MODELS ───────────────────────────────────────────────────────
results = {}
results['Latent_11'] = biomarker_panel_lasso('Latent_11', 0.25, top10_latent11)
results['Latent_5']  = biomarker_panel_lasso('Latent_5',  0.75, top10_latent5)
results['Latent_1']  = biomarker_panel_lasso('Latent_1',  0.25, top10_latent1)
results['T1_75p']    = biomarker_panel_lasso('T1_75th_Percentile', 0.75, top10_t1_75p)

# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("SUMMARY — 10-protein LASSO panels")
print(f"{'='*65}")
print(f"{'Phenotype':<25} {'CV AUC':<12} {'Test AUC':<10} {'95% CI'}")
print("-" * 65)
for key, res in results.items():
    print(f"{key:<25} {res['cv_auc']:.3f}±{res['cv_std']:.3f}   "
          f"{res['test_auc']:.3f}      "
          f"{res['ci_lower']:.3f}–{res['ci_upper']:.3f}")

In [ ]:
# ── TOP 5 PROTEIN PANELS ──────────────────────────────────────────────────────
top5_latent5  = top10_latent5[:5]
top5_latent11 = top10_latent11[:5]
top5_latent1  = top10_latent1[:5]
top5_t1_75p   = top10_t1_75p[:5]

results_top5 = {}
results_top5['Latent_11'] = biomarker_panel_lasso('Latent_11',          0.25, top5_latent11)
results_top5['Latent_5']  = biomarker_panel_lasso('Latent_5',           0.75, top5_latent5)
results_top5['Latent_1']  = biomarker_panel_lasso('Latent_1',           0.25, top5_latent1)
results_top5['T1_75p']    = biomarker_panel_lasso('T1_75th_Percentile', 0.75, top5_t1_75p)

# ── COMPARISON TABLE: TOP 5 vs TOP 10 ────────────────────────────────────────
print(f"\n{'='*75}")
print("COMPARISON — Top 5 vs Top 10 protein LASSO panels")
print(f"{'='*75}")
print(f"{'Phenotype':<25} {'Panel':<10} {'CV AUC':<15} {'Test AUC':<10} {'95% CI'}")
print("-" * 75)

for key in results:
    for label, res_dict in [('Top 10', results), ('Top 5', results_top5)]:
        res = res_dict[key]
        print(f"{key:<25} {label:<10} "
              f"{res['cv_auc']:.3f}±{res['cv_std']:.3f}      "
              f"{res['test_auc']:.3f}      "
              f"{res['ci_lower']:.3f}–{res['ci_upper']:.3f}")
    print()

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

def bnp_baseline(phenotype, cutoff_percentile):
    """
    Evaluate NT-proBNP alone as a single biomarker classifier
    for a given T1 phenotype and cutoff.
    """
    # Build dataset
    df = protein_data[['eid', 'nppb']].merge(   # update 'nppb' if different
        unet_t1, left_on='eid', right_on='Patient_ID', how='inner'
    ).merge(
        latent_dimensions, left_on='eid', right_on='IID', how='inner'
    ).dropna()

    threshold   = df[phenotype].quantile(cutoff_percentile)
    df['label'] = (df[phenotype] > threshold).astype(int)

    X = df[['nppb']].values
    y = df['label'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # AUC from raw NT-proBNP value as score
    auc = roc_auc_score(y_test, X_test)

    # Bootstrap CI
    bootstrap_aucs = []
    for _ in range(1000):
        idx = np.random.choice(len(y_test), len(y_test), replace=True)
        if len(np.unique(y_test[idx])) == 2:
            bootstrap_aucs.append(roc_auc_score(y_test[idx], X_test[idx]))
    ci_lower = np.percentile(bootstrap_aucs, 2.5)
    ci_upper = np.percentile(bootstrap_aucs, 97.5)

    print(f"{phenotype:<25} NT-proBNP AUC: {auc:.3f} (95% CI {ci_lower:.3f}–{ci_upper:.3f})")
    return auc, ci_lower, ci_upper


# Run for all phenotypes
print("NT-proBNP single biomarker performance:")
print("-" * 60)
for phenotype, cutoff in [
    ('Mean_T1',            0.75),
    ('Latent_11',          0.25),
    ('Latent_5',           0.75),
    ('Latent_1',           0.25),
    ('T1_75th_Percentile', 0.75)
]:
    bnp_baseline(phenotype, cutoff)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

def TNNI3_baseline(phenotype, cutoff_percentile):
    """
    Evaluate NT-proBNP alone as a single biomarker classifier
    for a given T1 phenotype and cutoff.
    """
    # Build dataset
    df = protein_data[['eid', 'tnni3']].merge(   # update 'tnni3' if different
        unet_t1, left_on='eid', right_on='Patient_ID', how='inner'
    ).merge(
        latent_dimensions, left_on='eid', right_on='IID', how='inner'
    ).dropna()

    threshold   = df[phenotype].quantile(cutoff_percentile)
    df['label'] = (df[phenotype] > threshold).astype(int)

    X = df[['tnni3']].values
    y = df['label'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # AUC from raw NT-proBNP value as score
    auc = roc_auc_score(y_test, X_test)

    # Bootstrap CI
    bootstrap_aucs = []
    for _ in range(1000):
        idx = np.random.choice(len(y_test), len(y_test), replace=True)
        if len(np.unique(y_test[idx])) == 2:
            bootstrap_aucs.append(roc_auc_score(y_test[idx], X_test[idx]))
    ci_lower = np.percentile(bootstrap_aucs, 2.5)
    ci_upper = np.percentile(bootstrap_aucs, 97.5)

    print(f"{phenotype:<25} Troponin AUC: {auc:.3f} (95% CI {ci_lower:.3f}–{ci_upper:.3f})")
    return auc, ci_lower, ci_upper


# Run for all phenotypes
print("Troponin single biomarker performance:")
print("-" * 60)
for phenotype, cutoff in [
    ('Mean_T1',            0.75),
    ('Latent_11',          0.25),
    ('Latent_5',           0.75),
    ('Latent_1',           0.25),
    ('T1_75th_Percentile', 0.75)
]:
    TNNI3_baseline(phenotype, cutoff)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def bnp_baseline_continuous(phenotype):
    """
    Evaluate NT-proBNP as a predictor of continuous T1 phenotype.
    """
    df = protein_data[['eid', 'nppb']].merge(
        unet_t1, left_on='eid', right_on='Patient_ID', how='inner'
    ).merge(
        latent_dimensions, left_on='eid', right_on='IID', how='inner'
    ).dropna()

    X = df[['nppb']].values
    y = df[phenotype].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Linear regression
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # Bootstrap CI for R²
    bootstrap_r2s = []
    for _ in range(1000):
        idx = np.random.choice(len(y_test), len(y_test), replace=True)
        try:
            bootstrap_r2s.append(r2_score(y_test[idx], y_pred[idx]))
        except:
            pass
    ci_lower = np.percentile(bootstrap_r2s, 2.5)
    ci_upper = np.percentile(bootstrap_r2s, 97.5)

    # Pearson correlation
    from scipy.stats import pearsonr
    r, p = pearsonr(X_test.flatten(), y_test)

    print(f"{phenotype:<25} R²={r2:.3f} (95% CI {ci_lower:.3f}–{ci_upper:.3f}) "
          f"RMSE={rmse:.1f} | Pearson r={r:.3f} p={p:.3e}")

    return r2, ci_lower, ci_upper, r, p


# Run for all continuous phenotypes
print("\nNT-proBNP continuous prediction:")
print("-" * 75)
for phenotype in [
    'Mean_T1',
    'T1_75th_Percentile',
    'Latent_11',
    'Latent_5',
    'Latent_1'
]:
    bnp_baseline_continuous(phenotype)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def tnni3_baseline_continuous(phenotype):
    """
    Evaluate Troponin as a predictor of continuous T1 phenotype.
    """
    df = protein_data[['eid', 'tnni3']].merge(
        unet_t1, left_on='eid', right_on='Patient_ID', how='inner'
    ).merge(
        latent_dimensions, left_on='eid', right_on='IID', how='inner'
    ).dropna()

    X = df[['tnni3']].values
    y = df[phenotype].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Linear regression
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # Bootstrap CI for R²
    bootstrap_r2s = []
    for _ in range(1000):
        idx = np.random.choice(len(y_test), len(y_test), replace=True)
        try:
            bootstrap_r2s.append(r2_score(y_test[idx], y_pred[idx]))
        except:
            pass
    ci_lower = np.percentile(bootstrap_r2s, 2.5)
    ci_upper = np.percentile(bootstrap_r2s, 97.5)

    # Pearson correlation
    from scipy.stats import pearsonr
    r, p = pearsonr(X_test.flatten(), y_test)

    print(f"{phenotype:<25} R²={r2:.3f} (95% CI {ci_lower:.3f}–{ci_upper:.3f}) "
          f"RMSE={rmse:.1f} | Pearson r={r:.3f} p={p:.3e}")

    return r2, ci_lower, ci_upper, r, p


# Run for all continuous phenotypes
print("\Troponin continuous prediction:")
print("-" * 75)
for phenotype in [
    'Mean_T1',
    'T1_75th_Percentile',
    'Latent_11',
    'Latent_5',
    'Latent_1'
]:
    tnni3_baseline_continuous(phenotype)

In [ ]:
import numpy as np
from scipy import stats

# ── RECONSTRUCT TEST SET ──────────────────────────────────────────────────────
df_binary = df[[phenotype] + sig_proteins + ['eid']].copy()
df_binary['label'] = (df_binary[phenotype] > df_binary[phenotype].quantile(0.75)).astype(int)
df_binary = df_binary.dropna().reset_index(drop=True)

X = df_binary[sig_proteins].values
y = df_binary['label'].values
idx = np.arange(len(df_binary))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=y)

test_eids = df_binary.iloc[test_idx]['eid'].values
test_labels = y[test_idx]

test_df = pd.DataFrame({'eid': test_eids, 'label': test_labels})
test_df = test_df.merge(big_pheno_data, left_on='eid', right_on='id', how='left')

# disease group membership
for group_name, group_ids in disease_groups.items():
    test_df[group_name] = test_df['eid'].isin(group_ids).astype(int)

pos = test_df[test_df['label'] == 1]
neg = test_df[test_df['label'] == 0]

print(f"N positive (high T1): {len(pos)}  |  N negative (low T1): {len(neg)}\n")

# ── CONTINUOUS PHENOTYPES ─────────────────────────────────────────────────────
continuous_cols = [
    'LVEF', 'LVEF (%)', 'RVEF (%)',
    'LVEDV', 'LVESV (mL)', 'LVSV (mL)', 'LVM (g)', 'iLVM (g/m2)',
    'LVCO (L/min)',
    'systolic_BP', 'diastolic_BP',
    'current_age'
]
continuous_cols = [c for c in continuous_cols if c in test_df.columns]

print(f"{'Variable':<25} {'Pos (mean±SD)':>20} {'Neg (mean±SD)':>20} {'p':>8}")
print("-" * 78)
for col in continuous_cols:
    p_vals = pos[col].dropna()
    n_vals = neg[col].dropna()
    if len(p_vals) < 3 or len(n_vals) < 3:
        continue
    _, pval = stats.mannwhitneyu(p_vals, n_vals, alternative='two-sided')
    sig = '*' if pval < 0.05 else ''
    print(f"{col:<25} {p_vals.mean():>8.2f}±{p_vals.std():>6.2f}   "
          f"{n_vals.mean():>8.2f}±{n_vals.std():>6.2f}   {pval:>8.3f} {sig}")

# ── DISEASE GROUP ENRICHMENT ──────────────────────────────────────────────────
print(f"\n{'Group':<25} {'Pos N (%)':>12} {'Neg N (%)':>12} {'p (Fisher)':>12}")
print("-" * 65)
for group_name in disease_groups.keys():
    if group_name not in test_df.columns:
        continue
    pos_n = pos[group_name].sum()
    neg_n = neg[group_name].sum()
    _, pval = stats.fisher_exact([[pos_n, len(pos) - pos_n],
                                  [neg_n, len(neg) - neg_n]])
    sig = '*' if pval < 0.05 else ''
    print(f"{group_name:<25} {pos_n:>5} ({100*pos_n/len(pos):>4.1f}%)  "
          f"{neg_n:>5} ({100*neg_n/len(neg):>4.1f}%)  {pval:>12.3f} {sig}")

# ── ICD-10 CODES ──────────────────────────────────────────────────────────────
# cardiomyopathy (I42), ischaemia (I20-I25), diabetes (E10-E14)
icd_patterns = {
    'Cardiomyopathy (I42)':  'I42',
    'Ischaemic HD (I20-I25)': '^I2[0-5]',
    'Diabetes (E10-E14)':     '^E1[0-4]',
    'Hypertension (I10)':     'I10',
    'Heart failure (I50)':    'I50',
}

icd_col = 'icd10' if 'icd10' in test_df.columns else 'mr_icd10'
print(f"\n── ICD-10 enrichment (col: {icd_col}) ──")
print(f"{'Diagnosis':<25} {'Pos N (%)':>12} {'Neg N (%)':>12} {'p (Fisher)':>12}")
print("-" * 65)

for label, pattern in icd_patterns.items():
    pos_n = pos[icd_col].fillna('').str.contains(pattern, regex=True).sum()
    neg_n = neg[icd_col].fillna('').str.contains(pattern, regex=True).sum()
    _, pval = stats.fisher_exact([[pos_n, len(pos) - pos_n],
                                  [neg_n, len(neg) - neg_n]])
    sig = '*' if pval < 0.05 else ''
    print(f"{label:<25} {pos_n:>5} ({100*pos_n/len(pos):>4.1f}%)  "
          f"{neg_n:>5} ({100*neg_n/len(neg):>4.1f}%)  {pval:>12.3f} {sig}")

In [ ]:
def test_reduced_panel(df, full_sig_proteins, reduced_panel, phenotype, cutoff_percentile=0.75):
    """
    Compare full PWAS protein panel vs reduced SHAP-selected panel.
    """
    from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score, roc_curve
    from xgboost import XGBClassifier
    import matplotlib.pyplot as plt
    import numpy as np

    # ── BUILD DATAFRAME ───────────────────────────────────────────────────────
    all_proteins = list(set(full_sig_proteins + reduced_panel))
    
    df_model = protein_data[['eid'] + all_proteins].merge(
        unet_t1, left_on='eid', right_on='Patient_ID', how='inner'
    ).dropna()
    df_model = df_model.merge(
        latent_dimensions, left_on='eid', right_on='IID', how='inner'
    ).dropna()

    # ── BINARY LABEL ──────────────────────────────────────────────────────────
    threshold = df_model[phenotype].quantile(cutoff_percentile)
    df_model['label'] = (df_model[phenotype] > threshold).astype(int)
    
    n_cases    = df_model['label'].sum()
    n_controls = (df_model['label'] == 0).sum()
    spw        = n_controls / n_cases

    # ── TRAIN/TEST SPLIT ──────────────────────────────────────────────────────
    # Use same random state so splits are identical for fair comparison
    X_full    = df_model[full_sig_proteins].values
    X_reduced = df_model[reduced_panel].values
    y         = df_model['label'].values

    X_full_train, X_full_test, y_train, y_test = train_test_split(
        X_full, y, test_size=0.2, random_state=42, stratify=y
    )
    X_red_train = X_full_train[:, [full_sig_proteins.index(p) for p in reduced_panel]]
    X_red_test  = X_full_test[:, [full_sig_proteins.index(p) for p in reduced_panel]]

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    results = {}
    for name, X_tr, X_te, n_feat in [
        ('Full panel',    X_full_train,  X_full_test,  len(full_sig_proteins)),
        ('Reduced panel', X_red_train,   X_red_test,   len(reduced_panel))
    ]:
        xgb = XGBClassifier(
            n_estimators=500, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=spw, random_state=42,
            n_jobs=-1, verbosity=0, eval_metric='auc'
        )
        xgb.fit(X_tr, y_train, eval_set=[(X_te, y_test)], verbose=False)
        
        proba    = xgb.predict_proba(X_te)[:, 1]
        auc      = roc_auc_score(y_test, proba)
        cv_aucs  = cross_val_score(xgb, 
                                   X_full if name == 'Full panel' else X_reduced,
                                   y, cv=cv, scoring='roc_auc', n_jobs=-1)
        
        results[name] = {
            'auc': auc, 'cv_mean': cv_aucs.mean(),
            'cv_std': cv_aucs.std(), 'proba': proba,
            'n_features': n_feat
        }
        print(f"{name} ({n_feat} proteins)")
        print(f"  Test AUC:    {auc:.3f}")
        print(f"  CV AUC:      {cv_aucs.mean():.3f} ± {cv_aucs.std():.3f}\n")

    # ── ROC COMPARISON PLOT ───────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 6))
    colors = ['#2980B9', '#C0392B']
    
    for (name, res), color in zip(results.items(), colors):
        fpr, tpr, _ = roc_curve(y_test, res['proba'])
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f"{name} — {res['n_features']} proteins "
                      f"(AUC={res['auc']:.3f}, CV={res['cv_mean']:.3f}±{res['cv_std']:.3f})")

    ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(
        f'Full vs Reduced Panel\n{phenotype} >{cutoff_percentile*100:.0f}th percentile',
        fontsize=11, fontweight='bold'
    )
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(f'roc_comparison_{phenotype}_{int(cutoff_percentile*100)}pct.png',
                dpi=300, bbox_inches='tight')
    plt.show()

    return results


# ── USAGE ─────────────────────────────────────────────────────────────────────
reduced_panel = ['fap', 'cd38', 'il1rl1', 'ren', 'angpt2',
                 'igfbp1', 'clec14a', 'wnt9a', 'ptprm', 'mmp8']

results = test_reduced_panel(
    df, sig_proteins, reduced_panel,
    phenotype='T1_75th_Percentile',
    cutoff_percentile=0.75
)

In [ ]:
! pip install imblearn

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import numpy as np

# ── 1. BUILD DATASET ──────────────────────────────────────────────────────────
CUTOFF = 1000

df_model = protein_data[['eid'] + sig_proteins].merge(
    unet_t1, left_on='eid', right_on='Patient_ID', how='inner'
).dropna()

df_model['label'] = (df_model['Mean_T1'] > CUTOFF).astype(int)

n_cases    = df_model['label'].sum()
n_controls = (df_model['label'] == 0).sum()
print(f"Before SMOTE — Cases: {n_cases}, Controls: {n_controls}")

X = df_model[sig_proteins].values
y = df_model['label'].values

# ── 2. TRAIN/TEST SPLIT BEFORE SMOTE ─────────────────────────────────────────
# CRITICAL: split BEFORE applying SMOTE
# SMOTE should only be applied to training data
# otherwise synthetic samples leak into test set -> inflated AUC
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train before SMOTE — Cases: {y_train.sum()}, Controls: {(y_train==0).sum()}")

# ── 3. APPLY SMOTE TO TRAINING SET ONLY ──────────────────────────────────────
# k_neighbors must be < n_minority_samples
k = min(5, y_train.sum() - 1)
smote = SMOTE(random_state=42, k_neighbors=k)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Train after SMOTE  — Cases: {y_train_sm.sum()}, Controls: {(y_train_sm==0).sum()}")
print(f"Test (unchanged)   — Cases: {y_test.sum()}, Controls: {(y_test==0).sum()}")

# ── 4. FIT MODEL ──────────────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0,
    eval_metric='auc'
    # Note: no scale_pos_weight needed since SMOTE balances the classes
)
xgb.fit(X_train_sm, y_train_sm,
        eval_set=[(X_test, y_test)],
        verbose=False)

# ── 5. EVALUATE ON REAL (UNSMOTED) TEST SET ───────────────────────────────────
proba = xgb.predict_proba(X_test)[:, 1]
auc   = roc_auc_score(y_test, proba)

print(f"\nTest AUC: {auc:.3f}")
print("\nClassification Report (real test set):")
print(classification_report(y_test, xgb.predict(X_test)))

# ── 6. ROC CURVE ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
fpr, tpr, _ = roc_curve(y_test, proba)
ax.plot(fpr, tpr, color='#C0392B', linewidth=2,
        label=f'XGBoost + SMOTE (AUC={auc:.3f})')
ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(f'ROC Curve — Mean T1 >{CUTOFF}ms\nSMOTE oversampling on training set',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('roc_smote_1000ms.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import shap
import numpy as np

shap_cv = np.zeros(len(sig_proteins))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for train_idx, val_idx in cv.split(X, y):
    xgb_fold = XGBClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=n_controls/n_cases,
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_fold.fit(X[train_idx], y[train_idx])
    explainer = shap.TreeExplainer(xgb_fold)
    sv = explainer.shap_values(X[val_idx])
    shap_cv += np.abs(sv).mean(axis=0)

shap_cv /= 5  # average across folds

# Rank proteins by CV-averaged SHAP
shap_df = pd.DataFrame({
    'protein': sig_proteins,
    'cv_shap': shap_cv
}).sort_values('cv_shap', ascending=False)

print(shap_df.head(20).to_string())

panel_sizes = [5, 10, 15, 20, 25, 30, 50]
aucs = []

# Use CV-averaged SHAP ranking
top_proteins_ranked = shap_df['protein'].tolist()

for n in panel_sizes:
    panel = top_proteins_ranked[:n]
    X_panel = df_model[panel].values
    
    cv_aucs = cross_val_score(
        XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05,
                      scale_pos_weight=n_controls/n_cases,
                      random_state=42, n_jobs=-1, verbosity=0),
        X_panel, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc', n_jobs=-1
    )
    aucs.append(cv_aucs.mean())
    print(f"Top {n:3d} proteins — CV AUC: {cv_aucs.mean():.3f} ± {cv_aucs.std():.3f}")

# Plot AUC vs panel size
plt.figure(figsize=(8, 5))
plt.plot(panel_sizes, aucs, 'o-', color='#C0392B', linewidth=2, markersize=8)
plt.axhline(y=max(aucs), color='grey', linestyle='--', alpha=0.5, label='Max AUC')
plt.xlabel('Number of proteins in panel', fontsize=12)
plt.ylabel('5-fold CV AUC', fontsize=12)
plt.title('Panel size vs predictive performance\nT1_75th_Percentile >75th percentile',
          fontsize=11, fontweight='bold')
plt.legend()
plt.spines if False else None
plt.tight_layout()
plt.savefig('panel_size_auc.png', dpi=300, bbox_inches='tight')
plt.show()

# Find minimum panel that stays within 0.02 of full model AUC
full_auc = cross_val_score(
    XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05,
                  scale_pos_weight=n_controls/n_cases,
                  random_state=42, n_jobs=-1, verbosity=0),
    X, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1
).mean()

print(f"\nFull panel CV AUC: {full_auc:.3f}")
for n, auc in zip(panel_sizes, aucs):
    if auc >= full_auc - 0.02:
        print(f"Minimum panel within 0.02 of full AUC: {n} proteins")
        break
        

In [ ]:
shap_df

In [ ]:
# Only run LASSO on top 50 CV-SHAP proteins rather than full panel
top50 = top_proteins_ranked[:50]
X_top50 = df_model[top50].values

scaler = StandardScaler()
X_top50_scaled = scaler.fit_transform(X_top50)

lasso = LogisticRegressionCV(
    Cs=10, cv=5, penalty='l1', solver='liblinear',
    scoring='roc_auc', random_state=42, n_jobs=-1
)
lasso.fit(X_top50_scaled, y)

lasso_cv_auc = cross_val_score(
    lasso, X_top50_scaled, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1
)
print(f"LASSO CV AUC: {lasso_cv_auc.mean():.3f} ± {lasso_cv_auc.std():.3f}")

selected = [top50[i] for i, c in enumerate(lasso.coef_[0]) if c != 0]
print(f"Proteins selected: {len(selected)}")
print(selected)

In [ ]:
# Rank LASSO proteins by coefficient magnitude
lasso_coef = pd.DataFrame({
    'protein': top50,  # or sig_proteins if run on full panel
    'coefficient': lasso.coef_[0]
}).query('coefficient != 0').assign(
    abs_coef=lambda x: x['coefficient'].abs()
).sort_values('abs_coef', ascending=False)

print(lasso_coef.to_string())

# Test reduced LASSO panels
lasso_ranked = lasso_coef['protein'].tolist()

print(f"\n{'Panel':<25} {'CV AUC':<10} {'SD'}")
print("-" * 45)

for n in [5, 10, 15, 20, 30, 48]:
    panel   = lasso_ranked[:n]
    X_panel = df_model[panel].values
    X_scaled = StandardScaler().fit_transform(X_panel)
    
    aucs = cross_val_score(
        LogisticRegressionCV(Cs=10, cv=5, penalty='l1',
                             solver='liblinear', scoring='roc_auc',
                             random_state=42),
        X_scaled, y,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc', n_jobs=-1
    )
    print(f"Top {n:<3} LASSO proteins    {aucs.mean():.3f}      ±{aucs.std():.3f}")

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

# ── TRAIN/TEST SPLIT ──────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42, stratify=y
)

fig, axes = plt.subplots(len(panel_sizes) + 1, 2, figsize=(14, (len(panel_sizes) + 1) * 4))
fig.suptitle('ROC Curves and Confusion Matrices\nT1_75th_Percentile >75th percentile',
             fontsize=14, fontweight='bold', y=1.01)

def plot_row(ax_roc, ax_cm, X_tr, X_te, label, color):
    xgb = XGBClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=n_controls/n_cases,
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb.fit(X_tr, y_train)
    proba = xgb.predict_proba(X_te)[:, 1]
    pred  = xgb.predict(X_te)
    auc   = roc_auc_score(y_test, proba)

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax_roc.plot(fpr, tpr, color=color, linewidth=2,
                label=f'AUC={auc:.3f}')
    ax_roc.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.4)
    ax_roc.set_xlabel('False Positive Rate', fontsize=10)
    ax_roc.set_ylabel('True Positive Rate', fontsize=10)
    ax_roc.set_title(label, fontsize=11, fontweight='bold')
    ax_roc.legend(fontsize=9)
    ax_roc.spines['top'].set_visible(False)
    ax_roc.spines['right'].set_visible(False)

    # Confusion matrix
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Low T1', 'High T1']
    )
    disp.plot(ax=ax_cm, colorbar=False, cmap='Blues')
    ax_cm.set_title(f'{label}\nFP={cm[0,1]}, FN={cm[1,0]}',
                    fontsize=11, fontweight='bold')

    # Print FP/FN
    tn, fp, fn, tp = cm.ravel()
    print(f"{label:<30} AUC={auc:.3f}  TP={tp}  TN={tn}  FP={fp}  FN={fn}  "
          f"Sensitivity={tp/(tp+fn):.2f}  Specificity={tn/(tn+fp):.2f}")
    
    return auc

# Colour palette
colors = plt.cm.tab10(np.linspace(0, 1, len(panel_sizes) + 1))

print(f"\n{'Panel':<30} {'AUC':<8} {'TP':<5} {'TN':<5} {'FP':<5} {'FN':<5} "
      f"{'Sensitivity':<13} {'Specificity'}")
print("-" * 80)

# Full panel
X_train_full = X_train
X_test_full  = X_test
plot_row(axes[0,0], axes[0,1],
         X_train_full, X_test_full,
         f'Full panel ({len(sig_proteins)} proteins)', colors[0])

# Reduced panels
for i, n in enumerate(panel_sizes):
    panel        = top_proteins_ranked[:n]
    protein_idx  = [sig_proteins.index(p) for p in panel]
    X_tr_panel   = X_train[:, protein_idx]
    X_te_panel   = X_test[:, protein_idx]
    
    plot_row(axes[i+1, 0], axes[i+1, 1],
             X_tr_panel, X_te_panel,
             f'Top {n} proteins', colors[i+1])

plt.tight_layout()
plt.savefig('roc_confusion_all_panels.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Final 30-protein panel
final_n           = 30
final_panel       = lasso_ranked[:final_n]
final_panel_idx   = [top50.index(p) for p in final_panel]

X_dev_final  = X_dev_scaled[:, final_panel_idx]
X_test_final = X_test_scaled[:, final_panel_idx]

final_lasso = LogisticRegressionCV(
    Cs=10, cv=5, penalty='l1', solver='liblinear',
    scoring='roc_auc', random_state=42, n_jobs=-1
)
final_lasso.fit(X_dev_final, y_dev)

# Test set AUC — reported once
test_proba = final_lasso.predict_proba(X_test_final)[:, 1]
test_auc   = roc_auc_score(y_test, test_proba)

print(f"Final panel:          {final_n} proteins")
print(f"Dev CV AUC:           {panel_dev_aucs[final_n].mean():.3f} ± {panel_dev_aucs[final_n].std():.3f}")
print(f"Test set AUC:         {test_auc:.3f}")
print(f"\nDev N:    {len(y_dev)} (cases: {y_dev.sum()})")
print(f"Test N:   {len(y_test)} (cases: {y_test.sum()})")

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap
import numpy as np
import pandas as pd

# ── 1. BUILD DATASET ──────────────────────────────────────────────────────────
phenotype         = 'T1_75th_Percentile'
cutoff_percentile = 0.75

df = protein_data[['eid'] + sig_proteins].merge(
    unet_t1, left_on='eid', right_on='Patient_ID', how='inner'
).merge(
    latent_dimensions, left_on='eid', right_on='IID', how='inner'
).dropna()

threshold   = df[phenotype].quantile(cutoff_percentile)
df['label'] = (df[phenotype] > threshold).astype(int)

X_all      = df[sig_proteins].values
y_all      = df['label'].values
n_cases    = y_all.sum()
n_controls = (y_all == 0).sum()
spw        = n_controls / n_cases

print(f"Total N:   {len(df)}")
print(f"Cases:     {n_cases} ({n_cases/len(df)*100:.1f}%)")
print(f"Controls:  {n_controls} ({n_controls/len(df)*100:.1f}%)")
print(f"Threshold: {threshold:.1f}ms")

# ── 2. HOLD OUT TEST SET ──────────────────────────────────────────────────────
X_dev, X_test, y_dev, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f"\nDev N:  {len(y_dev)} (cases: {y_dev.sum()})")
print(f"Test N: {len(y_test)} (cases: {y_test.sum()})")

# ── 3. CV SHAP ON DEV SET ─────────────────────────────────────────────────────
print("\nComputing CV SHAP on development set...")
shap_cv  = np.zeros(len(sig_proteins))
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_dev, y_dev)):
    xgb_fold = XGBClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw,
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_fold.fit(X_dev[train_idx], y_dev[train_idx])
    explainer = shap.TreeExplainer(xgb_fold)
    sv        = explainer.shap_values(X_dev[val_idx])
    shap_cv  += np.abs(sv).mean(axis=0)
    print(f"  Fold {fold+1}/5 done")

shap_cv /= 5
shap_df  = pd.DataFrame({
    'protein': sig_proteins,
    'cv_shap': shap_cv
}).sort_values('cv_shap', ascending=False).reset_index(drop=True)

top50     = shap_df['protein'].tolist()[:50]
top50_idx = [sig_proteins.index(p) for p in top50]

print(f"\nTop 10 by CV SHAP:")
print(shap_df.head(10).to_string())

# ── 4. LASSO ON DEV SET ───────────────────────────────────────────────────────
print("\nFitting LASSO on development set...")
scaler        = StandardScaler()
X_dev_top50   = X_dev[:, top50_idx]
X_test_top50  = X_test[:, top50_idx]
X_dev_scaled  = scaler.fit_transform(X_dev_top50)
X_test_scaled = scaler.transform(X_test_top50)

lasso = LogisticRegressionCV(
    Cs=10, cv=5, penalty='l1', solver='liblinear',
    scoring='roc_auc', random_state=42, n_jobs=-1
)
lasso.fit(X_dev_scaled, y_dev)

dev_cv_aucs = cross_val_score(
    lasso, X_dev_scaled, y_dev,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1
)
print(f"LASSO Dev CV AUC (48 proteins): {dev_cv_aucs.mean():.3f} ± {dev_cv_aucs.std():.3f}")

selected_mask = lasso.coef_[0] != 0
lasso_coef_df = pd.DataFrame({
    'protein':     top50,
    'coefficient': lasso.coef_[0]
}).query('coefficient != 0').assign(
    abs_coef=lambda x: x['coefficient'].abs()
).sort_values('abs_coef', ascending=False)

lasso_ranked = lasso_coef_df['protein'].tolist()
print(f"Proteins selected: {len(lasso_ranked)}")

# ── 5. PANEL SIZE OPTIMISATION ON DEV SET ────────────────────────────────────
print("\nTesting panel sizes on development set:")
panel_dev_aucs = {}

for n in [5, 10, 15, 20, 25, 30, 35, 40, len(lasso_ranked)]:
    panel       = lasso_ranked[:n]
    panel_idx   = [top50.index(p) for p in panel]
    X_dev_panel = X_dev_scaled[:, panel_idx]

    aucs = cross_val_score(
        LogisticRegressionCV(Cs=10, cv=5, penalty='l1',
                             solver='liblinear', scoring='roc_auc',
                             random_state=42),
        X_dev_panel, y_dev,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc', n_jobs=-1
    )
    panel_dev_aucs[n] = aucs
    print(f"  Top {n:<3} proteins — CV AUC: {aucs.mean():.3f} ± {aucs.std():.3f}")

# ── 6. FINAL EVALUATION ON TEST SET ──────────────────────────────────────────
final_n         = 30  # best panel from previous run
final_panel     = lasso_ranked[:final_n]
final_panel_idx = [top50.index(p) for p in final_panel]

X_dev_final  = X_dev_scaled[:, final_panel_idx]
X_test_final = X_test_scaled[:, final_panel_idx]

final_lasso = LogisticRegressionCV(
    Cs=10, cv=5, penalty='l1', solver='liblinear',
    scoring='roc_auc', random_state=42, n_jobs=-1
)
final_lasso.fit(X_dev_final, y_dev)

test_proba = final_lasso.predict_proba(X_test_final)[:, 1]
test_auc   = roc_auc_score(y_test, test_proba)

print(f"\n{'='*55}")
print(f"FINAL RESULTS")
print(f"{'='*55}")
print(f"Phenotype:            {phenotype} >{cutoff_percentile*100:.0f}th percentile ({threshold:.1f}ms)")
print(f"Total N:              {len(df)}")
print(f"Dev N:                {len(y_dev)} (cases: {y_dev.sum()})")
print(f"Test N:               {len(y_test)} (cases: {y_test.sum()})")
print(f"Full LASSO Dev CV AUC:{dev_cv_aucs.mean():.3f} ± {dev_cv_aucs.std():.3f}")
print(f"Top 30 Dev CV AUC:    {panel_dev_aucs[30].mean():.3f} ± {panel_dev_aucs[30].std():.3f}")
print(f"Test set AUC:         {test_auc:.3f}  <- independent validation")
print(f"\nFinal 30-protein panel:")
print(lasso_coef_df.head(30).to_string())

In [ ]:
# XGBoost on held-out test set
xgb_final = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=n_controls/n_cases,
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_final.fit(X_dev, y_dev)
xgb_test_proba = xgb_final.predict_proba(X_test)[:, 1]
xgb_test_auc   = roc_auc_score(y_test, xgb_test_proba)

# Bootstrap CI
bootstrap_aucs = []
for _ in range(1000):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    if len(np.unique(y_test[idx])) == 2:
        bootstrap_aucs.append(roc_auc_score(y_test[idx], xgb_test_proba[idx]))

ci_lower = np.percentile(bootstrap_aucs, 2.5)
ci_upper = np.percentile(bootstrap_aucs, 97.5)

print(f"XGBoost full panel:")
print(f"  Dev CV AUC:  {full_auc:.3f}")
print(f"  Test AUC:    {xgb_test_auc:.3f}")
print(f"  95% CI:      {ci_lower:.3f} – {ci_upper:.3f}")

In [ ]:
from sklearn.model_selection import KFold

for n in [3, 5, 7, 10, 15, 20, 30]:
    panel       = lasso_ranked[:n]
    panel_idx   = [sig_proteins.index(p) for p in panel]
    X_dev_p     = X_dev[:, panel_idx]
    X_test_p    = X_test[:, panel_idx]
    
    xgb_p = XGBClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=n_controls/n_cases,
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_p.fit(X_dev_p, y_dev)
    
    cv_auc   = cross_val_score(
        xgb_p, X_dev_p, y_dev,
        cv=KFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc', n_jobs=-1
    ).mean()
    test_auc = roc_auc_score(y_test, xgb_p.predict_proba(X_test_p)[:,1])
    
    print(f"Top {n:<3} proteins — CV AUC: {cv_auc:.3f} | Test AUC: {test_auc:.3f}")

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

# ── CONTINUOUS T1 PREDICTION ──────────────────────────────────────────────────
y_continuous = df['T1_75th_Percentile'].values

X_dev_c, X_test_c, y_dev_c, y_test_c = train_test_split(
    X_all, y_continuous,
    test_size=0.2, random_state=42
)

# Full panel
xgb_reg = XGBRegressor(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_reg.fit(X_dev_c, y_dev_c)

test_pred = xgb_reg.predict(X_test_c)
test_r2   = r2_score(y_test_c, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test_c, test_pred))

cv_r2 = cross_val_score(
    xgb_reg, X_dev_c, y_dev_c,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='r2', n_jobs=-1
)

print(f"Full panel ({len(sig_proteins)} proteins)")
print(f"  Dev CV R²:  {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")
print(f"  Test R²:    {test_r2:.3f}")
print(f"  Test RMSE:  {test_rmse:.1f}ms")

# 30-protein panel
panel_idx   = [sig_proteins.index(p) for p in final_panel]
X_dev_p     = X_dev_c[:, panel_idx]
X_test_p    = X_test_c[:, panel_idx]

xgb_reg_p = XGBRegressor(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)


In [ ]:
xgb_reg_p.fit(X_dev_p, y_dev_c)  

test_pred_p = xgb_reg_p.predict(X_test_p)
test_r2_p   = r2_score(y_test_c, test_pred_p)
test_rmse_p = np.sqrt(mean_squared_error(y_test_c, test_pred_p))

cv_r2_p = cross_val_score(
    xgb_reg_p, X_dev_p, y_dev_c,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='r2', n_jobs=-1
)

print(f"\n30-protein panel")
print(f"  Dev CV R²:  {cv_r2_p.mean():.3f} ± {cv_r2_p.std():.3f}")
print(f"  Test R²:    {test_r2_p:.3f}")
print(f"  Test RMSE:  {test_rmse_p:.1f}ms")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get predictions on test set
test_pred_p = xgb_reg_p.predict(X_test_p)

# Also get out-of-fold predictions on dev set
from sklearn.model_selection import KFold
oof_pred = np.zeros(len(y_dev_c))
for train_idx, val_idx in KFold(n_splits=5, shuffle=True, random_state=42).split(X_dev_p):
    m = XGBRegressor(
        n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0
    )
    m.fit(X_dev_p[train_idx], y_dev_c[train_idx])
    oof_pred[val_idx] = m.predict(X_dev_p[val_idx])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_true, y_pred, label, color, r2 in zip(
    axes,
    [y_dev_c,    y_test_c],
    [oof_pred,   test_pred_p],
    ['Dev set (out-of-fold)', 'Independent test set'],
    ['#2980B9',  '#C0392B'],
    [r2_score(y_dev_c, oof_pred), test_r2_p]
):
    ax.scatter(y_true, y_pred, alpha=0.3, s=15, color=color, rasterized=True)
    
    # Perfect prediction line
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'k--', linewidth=1, alpha=0.6, label='Perfect prediction')
    
    # Regression line
    m_fit, b_fit = np.polyfit(y_true, y_pred, 1)
    ax.plot(lims, [m_fit*x + b_fit for x in lims],
            color=color, linewidth=2, alpha=0.8,
            label=f'Fit (slope={m_fit:.2f})')
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ax.set_xlabel('Actual T1_75th_Percentile (ms)', fontsize=12)
    ax.set_ylabel('Predicted T1_75th_Percentile (ms)', fontsize=12)
    ax.set_title(f'{label}\nR²={r2:.3f}, RMSE={rmse:.1f}ms',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('30-protein panel — Predicted vs Actual T1_75th_Percentile',
             fontsize=13, fontweight='bold')
plt.tight_layout()
#plt.savefig('scatter_regression_30proteins.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for col in ['Mean_T1', 'T1_25th_Percentile', 'T1_50th_Percentile', 'T1_75th_Percentile', 'T1_99th_Percentile']:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} unique values")